In [1]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col, trim, lower, regexp_replace
import os

In [2]:
#exemple of path_data
path_landing = "../../../data/interim/delta_lake" 

In [3]:
mongo_connector_jar = "/home/provira/Documents/TFM/TFM/notebooks/P2/trusted_zone/jars/mongo-spark-connector_2.12-3.0.1.jar"
mongo_driver_jar = "/home/provira/Documents/TFM/TFM/notebooks/P2/trusted_zone/jars/mongo-java-driver-3.12.10.jar"

In [4]:
builder = SparkSession.builder \
    .appName("Trusted_Zone_Streaming") \
    .config("spark.jars", f"{mongo_connector_jar},{mongo_driver_jar}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

25/10/23 11:41:11 WARN Utils: Your hostname, provira-ERAZER-P6705-MD61203 resolves to a loopback address: 127.0.1.1; using 192.168.1.56 instead (on interface enp3s0f1)
25/10/23 11:41:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/provira/.ivy2/cache
The jars for the packages stored in: /home/provira/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-72b94b75-767e-466a-8718-dee3996c0b85;1.0
	confs: [default]
	found io.delta#delta-core_2.12;1.0.0 in central
	found org.antlr#antlr4;4.7 in central
	found org.antlr#antlr4-runtime;4.7 in central
	found org.antlr#antlr-runtime;3.5.2 in central
	found org.antlr#ST4;4.0.8 in central
	found org.abego.treelayout#org.abego.treelayout.core;1.0.3 in central
	found org.glassfish#javax.json;1.0.4 in central
	found com.ibm.icu#icu4j;58.2 in central
:: resolution report :: resolve 508ms :: artifacts dl 9ms
	:: modules in use:
	com.ibm.icu#icu4j;58.2 from central in [default]
	io.delta#delta-core_2.12;1.0.0 from central in [default]
	org.abego.treelayout#org.abego.treelayout.core;1.0.3 from central in [default]
	org.antlr#ST4;4.0.8 from central in [default]
	org.antlr#antlr-ru

In [5]:
spark.sparkContext._jsc.sc().listJars()

JavaObject id=o53

In [6]:
def readFromDeltaLake(path_landing):
    print(path_landing)
    return spark.read.format("delta").load(path_landing)

# Preprocessing

In [47]:
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import col, from_json

def parseDF(df_csv):
    #print(df_csv.columns)
    #print(df_csv.head())

    # Eliminar filas vacías
    df_csv_clean = df_csv.dropna(subset=["value"])

    # Definir el esquema del JSON dentro de la columna `value`
    value_schema = StructType() \
        .add("source", StringType()) \
        .add("author", StringType()) \
        .add("message", StringType()) \
        .add("published_at", StringType())

    # Convertir bytearray → string → JSON struct
    df_parsed = (
        df_csv_clean
        .withColumn("value_str", col("value").cast("string"))
        .withColumn("json", from_json(col("value_str"), value_schema))
        .select(
            col("json.source"),
            col("json.author"),
            col("json.message").alias("text"),     # 👈 renombrar aquí
            col("json.published_at")
        )
    )

    df_parsed.show(5, truncate=False)
    return df_parsed



In [35]:
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import col, from_json, decode

def cleanCSV(df_csv):

    df_csv = parseDF(df_csv)

    df_csv_clean = df_csv.dropna()

    rename_map = {
        "key": "id",
        "value": "emotion",
    }

    for old, new in rename_map.items():
        if old in df_csv_clean.columns:
            df_csv_clean = df_csv_clean.withColumnRenamed(old, new)
    print(df_csv_clean.head())
    # Only apply text processing if 'text' exists
    if "text" in df_csv_clean.columns:
        df_csv_clean = df_csv_clean \
            .withColumn("text", trim(col("text"))) \
            .withColumn("text", lower(col("text"))) \
            .withColumn("text", regexp_replace(col("text"), r"\bi m\b", "i'm")) \
            .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z0-9\s']", ""))  # keep letters, digits, spaces, apostrophes
    return df_csv_clean

In [36]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

#nltk.download('punkt')
#nltk.download('stopwords')
def tokenizer(df_csv_clean):
    stop_words = set(stopwords.words('english'))

    # Tokenizar
    tokenizer = Tokenizer(inputCol="text", outputCol="words")
    df_words = tokenizer.transform(df_csv_clean) #.limit(1000) limited to 1000 rows for performance

    # Eliminar stopwords (solo en inglés por defecto, pero puedes pasar las tuyas)
    remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    remover.setStopWords(list(stop_words))
    df_filtered = remover.transform(df_words)

    #Stemming
    stemmer = SnowballStemmer("english")

    def stem_tokens(tokens):
        return [stemmer.stem(token) for token in tokens]
    
    stem_udf = udf(stem_tokens, ArrayType(StringType()))

    df_stemmed = df_filtered.withColumn("stemmed_words", stem_udf("filtered_words"))

    #df_stemmed.select("text", "filtered_words", "stemmed_words").show(truncate=False)

    return df_stemmed
    



In [37]:
from pyspark.ml.feature import CountVectorizer, IDF

def tf_idf(df_stemmed):

    # Paso 1: Crear el CountVectorizer para extraer el vocabulario y conteo de tokens
    cv = CountVectorizer(inputCol="stemmed_words", outputCol="raw_features")
    cv_model = cv.fit(df_stemmed)             # Entrenas el modelo con el vocabulario
    df_featurized = cv_model.transform(df_stemmed)  # Transformas el DataFrame

    # Paso 2: Calcular TF-IDF a partir del conteo
    idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
    idf_model = idf.fit(df_featurized)          # Ajustar IDF sobre los datos
    df_tfidf = idf_model.transform(df_featurized) # Transformar con TF-IDF

    # Mostrar resultados
    #df_tfidf.select("stemmed_words", "raw_features", "tfidf_features").show(truncate=False)
    df_tfidf.printSchema()

    return df_tfidf


# Store in MongoDB

In [38]:

from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.linalg import VectorUDT

def df_clean(df_tfidf):
    def vector_to_array(v):
        return v.toArray().tolist() if v else None

    vector_to_array_udf = udf(vector_to_array, ArrayType(FloatType()))

    df_tfidf_safe = df_tfidf \
        .withColumn("raw_features_array", vector_to_array_udf("raw_features")) \
        .withColumn("tfidf_features_array", vector_to_array_udf("tfidf_features"))
    return df_tfidf_safe


In [53]:
def storeInMongoDB(df_tfidf_safe):
    df_tfidf_safe.select(
        "source", "author", "text", "published_at", "words", "filtered_words", "stemmed_words",
        "raw_features_array", "tfidf_features_array"
    ).write \
        .format("mongo") \
        .option("uri", "mongodb://localhost:27017") \
        .option("database", "tfm-trusted-zone-streaming") \
        .option("collection", "tf-idf") \
        .mode("append") \
        .save()

# Pipeline

In [40]:
cleanCSV_udf = udf(cleanCSV, ArrayType(StringType()))
tokenizer_udf = udf(tokenizer, ArrayType(StringType()))
df_clean_udf = udf(df_clean, ArrayType(StringType()))  # Update return type if it's different

In [41]:
types = ['csv', 'parquet', 'json', 'txt']
sources = ['Kaggle', 'uci', 'AWS']

In [ ]:
path_data = "./../../../data/interim/"

for source in sources:
    path = os.path.join(path_data, source)

    if os.path.isdir(path):
        for folder in os.listdir(path):
            full_path = os.path.join(path, folder)
            #print(f"Processing folder: {full_path}/")

            try:
                full_path = path_landing
                print(f"Processing folder: {full_path}/")
                df_csv = readFromDeltaLake(full_path)

                df_csv_clean = cleanCSV(df_csv) #  .limit(10) Limit to 10 rows for performance
                df_stemmed = tokenizer(df_csv_clean)
                df_tfidf = tf_idf(df_stemmed)
                df_tfidf_safe = df_clean(df_tfidf)
                df_tfidf_safe.show(5, truncate=False)
                storeInMongoDB(df_tfidf_safe)

            except Exception as e:
                print(f"Error processing folder {full_path}: {e}")

Processing folder: ../../../data/interim/delta_lake/
../../../data/interim/delta_lake


+------------+-------------------------------+-----------------------------------------+--------------------------------+
|source      |author                         |text                                     |published_at                    |
+------------+-------------------------------+-----------------------------------------+--------------------------------+
|youtube_live|@bogdannk9339                  |3-0?                                     |2025-10-19T06:24:03.998495+00:00|
|youtube_live|@AndreiBejinar                 |It’s 0-3                                 |2025-10-19T06:24:23.569528+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|amor da tia sorte...te amoooo            |2025-10-19T06:29:48.546666+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|bravooo claudio😂😂👏👏👏                |2025-10-19T06:31:36.687067+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|bueno claudio 0-9a tia vai dormir😂😂😂😂|2025-10-19T06:41:49.688981+00:00|
+------------+-------------------

root
 |-- source: string (nullable = true)
 |-- author: string (nullable = true)
 |-- text: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- stemmed_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- raw_features: vector (nullable = true)
 |-- tfidf_features: vector (nullable = true)



+------------+-------------------------------+--------------------------------+--------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+----------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Processing folder: ../../../data/interim/delta_lake/
../../../data/interim/delta_lake
+------------+-------------------------------+-----------------------------------------+--------------------------------+
|source      |author                         |text                                     |published_at                    |
+------------+-------------------------------+-----------------------------------------+--------------------------------+
|youtube_live|@bogdannk9339                  |3-0?                                     |2025-10-19T06:24:03.998495+00:00|
|youtube_live|@AndreiBejinar                 |It’s 0-3                                 |2025-10-19T06:24:23.569528+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|amor da tia sorte...te amoooo            |2025-10-19T06:29:48.546666+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|bravooo claudio😂😂👏👏👏                |2025-10-19T06:31:36.687067+00:00|
|youtube_live|@mariadelcielomendesribeiro2062|bueno claudio 0-9a 

root
 |-- source: string (nullable = true)
 |-- author: string (nullable = true)
 |-- text: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- stemmed_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- raw_features: vector (nullable = true)
 |-- tfidf_features: vector (nullable = true)



+------------+-------------------------------+--------------------------------+--------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+----------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Traceback (most recent call last):) / 1][Stage 623:>                (0 + 1) / 1]
  File "/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 192, in manager
KeyboardInterrupt
25/10/23 12:03:00 ERROR Executor: Exception in task 0.0 in stage 581.0 (TID 9109)
org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:550)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:539)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.sql.execution.python.PythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:89)
	at org.apache.spark.sql.execution.python.PythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:67)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:470)

KeyboardInterrupt: 